# 10 · Самостоятельная геометрия двух видов
Вы реализуете линейную геометрию и робастную оценку, затем восстановите позу и 3D с явными ограничениями масштаба. План:55 минут в аудитории на первые реализации и 90–130 минут дома на завершение, val/test и отчет. ВычисленияCPU, без обучения/скачивания. Пять TODO намеренно останавливают незаполненный шаблон.

**Гипотеза до запуска:** как threshold влияет на precision/recall; почему малый базис способен оставить малую эпиполярную ошибку при плохой глубине? Запишите ожидаемый результат.

## Координаты и смысл данных

`(u,v)` — пиксели, массив изображения индексируется `[v,u]`. Оси камеры: x вправо, y вниз, z вперед. `Xc = R @ Xworld + t`; R,t переводят **мир в камеру**. Центр камеры `C = -R.T @ t`. Проекция: `s*[u,v,1] = K @ Xc`. Нужна положительная глубина. Нет дисторсии; все сцены созданы кодом.

K: fx=fy=800 px, cx=320, cy=240; изображение 640×480. В эксперименте 160 точек имеют разные глубины 4…7.5 м, baseline .5 м, yaw камеры 2=.04rad, независимый Gaussian noise σ=.7px на каждую координату каждого кадра и 25%замененных target-наблюдений. GT-inlier означает, что соответствие не заменялось, а не что текущая модель согласуется с ним. Видимость обоих кадров проверяется до добавления шума.

Это неплоская 3D-сцена с известными камерами. Гомография плоского изображения из предыдущей работы не используется как датасет двух видов объемной сцены. Нет скачивания данных или весов.

In [ ]:
import importlib.util,subprocess,sys
requirements={'numpy':'numpy>=1.26,<3','cv2':'opencv-python-headless==4.12.0.88','matplotlib':'matplotlib>=3.8,<4','ipywidgets':'ipywidgets>=8,<9','threadpoolctl':'threadpoolctl>=3.5,<4'}
missing=[spec for module,spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import cv2
# В текущем Colab штатные варианты OpenCV конфликтуют: cv2 5 не содержит AKAZE.
# Исправляем только одноразовую среду Colab; локальную среду не меняем.
try:
    import google.colab as _colab_runtime
except ImportError:
    _colab_runtime=None
if _colab_runtime is not None and (cv2.__version__.split('.')[0]!='4' or not hasattr(cv2,'AKAZE_create')):
    import importlib.metadata as _opencv_metadata
    _variants=[]
    for _variant in ['opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless']:
        try:_opencv_metadata.version(_variant);_variants.append(_variant)
        except _opencv_metadata.PackageNotFoundError:pass
    if _variants:subprocess.check_call([sys.executable,'-m','pip','uninstall','-y',*_variants])
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps','opencv-python-headless==4.12.0.88'])
    raise RuntimeError('OpenCV 4.12 установлен. В Colab выберите «Среда выполнения → Перезапустить сеанс», затем выполните ноутбук сверху.')
if cv2.__version__.split('.')[0]!='4':
    import importlib.metadata as meta
    installed=[]
    for variant in ['opencv-python','opencv-python-headless','opencv-contrib-python','opencv-contrib-python-headless']:
        try:meta.version(variant);installed.append(variant)
        except meta.PackageNotFoundError:pass
    if len(installed)==1:instruction=f'В отдельной ячейке выполните %pip install {installed[0]}==4.12.0.88 и перезапустите ядро; сохраняйте один вариант OpenCV.'
    else:instruction='Создайте чистую среду с одним вариантом opencv-python-headless==4.12.0.88 и выберите ее kernel.'
    raise RuntimeError('Для воспроизводимого протокола нужен OpenCV4.x. '+instruction)
from threadpoolctl import threadpool_limits
_thread_limit=threadpool_limits(limits=1)
try:
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:pass
%matplotlib inline

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import json,csv,time,platform,math
cv2.setNumThreads(1)
plt.rcParams.update({'figure.dpi':105,'font.size':10})
K0=np.array([[800.,0,320],[0,800.,240],[0,0,1.]])
SIZE=(640,480) # width,height

def skew(t):
    x,y,z=np.asarray(t).ravel();return np.array([[0,-z,y],[z,0,-x],[-y,x,0.]])

def camera(K,R,C):
    t=-R@np.asarray(C);return K@np.c_[R,t],t

def scene(seed=0,noise=.7,outlier_fraction=.25,baseline=.5,yaw=.04,kind='general',n=160):
    """Authored 3D synthetic observations. No detector, real photo or measured camera pose."""
    rng=np.random.default_rng(seed)
    X=rng.uniform([-1.,-.75,4.],[1.,.75,7.5],size=(n*5,3))
    if kind=='planar':X[:,2]=5.
    if kind=='pure_rotation':baseline=0.
    R,_=cv2.Rodrigues(np.array([0.,yaw,0.]));C=np.array([baseline,0.,0.])
    P1,t1=camera(K0,np.eye(3),np.zeros(3));P2,t2=camera(K0,R,C)
    p=cv2.projectPoints(X,np.zeros(3),t1,K0,None)[0][:,0]
    q=cv2.projectPoints(X,cv2.Rodrigues(R)[0],t2,K0,None)[0][:,0]
    visible=(p[:,0]>=0)&(p[:,0]<640)&(p[:,1]>=0)&(p[:,1]<480)&(q[:,0]>=0)&(q[:,0]<640)&(q[:,1]>=0)&(q[:,1]<480)
    X=X[visible][:n];p=p[visible][:n];q=q[visible][:n]
    if len(X)!=n:raise ValueError('Not enough jointly visible points for these cameras')
    clean1=p.copy();clean2=q.copy();p=p+rng.normal(0,noise,p.shape);q=q+rng.normal(0,noise,q.shape)
    inlier=np.ones(n,bool);bad=rng.choice(n,round(outlier_fraction*n),replace=False)
    q[bad]=rng.uniform([0,0],SIZE,(len(bad),2));inlier[bad]=False
    F=np.linalg.inv(K0).T@skew(t2)@R@np.linalg.inv(K0)
    return dict(X=X,p=p,q=q,clean1=clean1,clean2=clean2,inlier=inlier,R=R,t=t2,C=C,K=K0.copy(),P1=P1,P2=P2,F=F,noise=noise,kind=kind,baseline=baseline,seed=seed)

def native_sampson(F,p,q):
    if F is None or not np.isfinite(F).all() or np.linalg.norm(F)<1e-14:return np.full(len(p),np.inf)
    return np.sqrt(np.array([cv2.sampsonDistance(np.r_[a,1.].astype(float),np.r_[b,1.].astype(float),F) for a,b in zip(p,q)]))

def ray_angles(p,q,K,R):
    rays1=np.c_[p,np.ones(len(p))]@np.linalg.inv(K).T
    rays2=(np.c_[q,np.ones(len(q))]@np.linalg.inv(K).T)@R
    rays1/=np.linalg.norm(rays1,axis=1,keepdims=True);rays2/=np.linalg.norm(rays2,axis=1,keepdims=True)
    return np.degrees(np.arccos(np.clip(np.sum(rays1*rays2,axis=1),-1,1)))

def draw_scene(ax,s,selected=0):
    X=s['X'];ax.scatter(X[:,0],X[:,2],X[:,1],c=X[:,2],cmap='viridis',s=9,alpha=.65)
    for C,R,col,label in [(np.zeros(3),np.eye(3),'#2563eb','C1'),(s['C'],s['R'],'#ef4444','C2')]:
        corners=np.array([[0.,0,1],[640,0,1],[640,480,1],[0,480,1],[0,0,1]])
        rays=(corners@np.linalg.inv(s['K']).T*.6)@R+C
        ax.plot(rays[:,0],rays[:,2],rays[:,1],color=col)
        for Q in rays[:4]:ax.plot([C[0],Q[0]],[C[2],Q[2]],[C[1],Q[1]],color=col,lw=.8)
        Q=X[selected];ax.plot([C[0],Q[0]],[C[2],Q[2]],[C[1],Q[1]],color=col,ls='--',lw=1)
        ax.text(C[0],C[2],C[1],label,color=col)
    ax.set(xlabel='X (m)',ylabel='Z (m)',zlabel='Y down (m)',title='Known scene and viewing rays')
    from matplotlib.ticker import MaxNLocator
    for axis in (ax.xaxis,ax.yaxis,ax.zaxis):axis.set_major_locator(MaxNLocator(3))
    ax.tick_params(labelsize=7,pad=1)
    ax.invert_zaxis();ax.view_init(elev=20,azim=-65)

def image_axes(ax,title):
    ax.set(xlim=(0,640),ylim=(480,0),aspect='equal',xlabel='u (px)',ylabel='v (px)',title=title)

def draw_epiline(ax,line,color='#ef4444'):
    a,b,c=line;points=[]
    if abs(b)>1e-12:
        for u in [0,640]:
            v=-(a*u+c)/b
            if 0<=v<=480:points.append([u,v])
    if abs(a)>1e-12:
        for v in [0,480]:
            u=-(b*v+c)/a
            if 0<=u<=640:points.append([u,v])
    if len(points)>=2:points=np.array(points[:2]);ax.plot(points[:,0],points[:,1],color=color,lw=1.7)

def save_csv(path,rows):
    keys=list(dict.fromkeys(k for r in rows for k in r))
    with Path(path).open('w',newline='') as f:
        writer=csv.DictWriter(f,keys);writer.writeheader();writer.writerows(rows)
print('OpenCV',cv2.__version__,'; OpenCV threads',cv2.getNumThreads())

## TODO1. `project_points`

Покажите на ручном примере, почему center C не равен t. Не используйте cv2.projectPoints в своей реализации.

До реализации придумайте собственный маленький вход и вручную запишите ответ; после реализации проверьте его кодом. Не копируйте только открытые тесты.

In [ ]:
def project_points(X,K,R,t):
    """Nx3 world points -> (Nx2 pixels, N camera_depth, N valid). Xc=R X+t; valid requires finite projection and depth>1e-12. Invalid pixels=NaN; empty supported. K/R wrong shape ->ValueError."""
    raise NotImplementedError('TODO1: world-to-camera and projection')

## TODO2. `eight_point`

Самостоятельно соберите линейную систему. SVD на 8×9 требует полного правого базиса, иначе можно потерять одномерное ядро. После обнуления третьего сингулярного числа F верните исходные пиксельные координаты. Это восьмиточечный учебный решатель; минимальный решатель F может использовать 7 точек.

До реализации придумайте собственный маленький вход и вручную запишите ответ; после реализации проверьте его кодом. Не копируйте только открытые тесты.

In [ ]:
def eight_point(p,q):
    """>=8 Nx2 correspondences -> F with x2.T F x1=0, Frobenius norm1, rank2. Hartley normalization: subtract centroid, scale mean radial distance to sqrt2. Build A, solve nullspace, enforce rank2, denormalize. Reject <8/nonfinite/coincident/rank(A)<8 via ValueError. Use full_matrices=True for exactly8 rows."""
    raise NotImplementedError('TODO2: normalized eight-point F')

## TODO3. `sampson_error`

Эта функция возвращает корень ошибки, а не квадрат. Поэтому сравнивайте с threshold в пикселях, а не с threshold². Масштаб F не должен менять результат. Нулевая F не должна давать нулевую ошибку всем парам.

До реализации придумайте собственный маленький вход и вручную запишите ответ; после реализации проверьте его кодом. Не копируйте только открытые тесты.

In [ ]:
def sampson_error(F,p,q):
    """Return N sqrt-Sampson errors in pixels: abs(x2.T F x1)/sqrt((Fx1)_x^2+(Fx1)_y^2+(F.T x2)_x^2+(F.T x2)_y^2). First divide F by its max absolute coefficient for scale-stable numerics; zero/nonfinite F ->inf. Undefined/nonfinite denominator<=1e-20 ->inf, never0. Empty supported; shape mismatch ->ValueError."""
    raise NotImplementedError('TODO3: Sampson error and units')

## TODO4. `ransac_f`

Бюджетная формула предполагает независимые попытки и использует оцененную долю w. Для конечной выборки без возвращения w⁸—приближение; объявленная confidence не гарантия при неверной модели, зависимых данных или cap. GT-метку inlier нельзя передавать решателю.

До реализации придумайте собственный маленький вход и вручную запишите ответ; после реализации проверьте его кодом. Не копируйте только открытые тесты.

In [ ]:
def ransac_f(p,q,threshold=1.,confidence=.999,max_trials=1500,seed=41):
    """Return (F_or_None, N bool consensus, info{trials,budget_limit,estimated_required,found}). Random seeded samples of8 without replacement, skip degenerate fits, score all pairs by sqrt Sampson<=threshold. Maximize inlier count then minimize median accepted error. Refit>=8 consensus points with eight_point (up to2 local updates), keep only a non-worse score. Update approximate trial limit ceil(log(1-confidence)/log(1-w**8)) from best observed inlier fraction; w=1 ->1, w=0 ->max_trials; cap. Stop at cap or adaptive limit. <8 or invalid settings ->ValueError; all degenerate ->None,empty consensus."""
    raise NotImplementedError('TODO4: robust F and stopping rule')

## TODO5. `triangulate_dlt`

Для каждого кадра используйте две строки вида u P[2]−P[0], v P[2]−P[1]. После деления на w проверьте глубину через project_points. Одинаковые камеры/лучи не определяют глубину; не заменяйте бесконечность произвольной большой Z.

До реализации придумайте собственный маленький вход и вручную запишите ответ; после реализации проверьте его кодом. Не копируйте только открытые тесты.

In [ ]:
def triangulate_dlt(P1,P2,p,q):
    """3x4 canonical pixel projection matrices and Nx2 pairs -> (Nx3 X,N finite). For each pair solve4x4 homogeneous linear equations via SVD. rank(A)<3 or |homogeneous w|<=1e-12 ->NaN/False. finite means finite intersection, NOT positive depth. Empty supported; shape mismatch ->ValueError."""
    raise NotImplementedError('TODO5: DLT triangulation')

## Открытые проверки

Они проверяют договоры функций, но не заменяют ваши 5 ручных примеров и объяснения.

In [ ]:
# 1. Lecture numerical oracle and non-visible/empty cases.
p,z,v=project_points([[1,.5,5]],K0,np.eye(3),[0,0,0]);assert np.allclose(p,[[480,320]]) and v[0]
q,_,_=project_points([[1,.5,5]],K0,np.eye(3),[-.5,0,0]);assert np.allclose(q,[[400,320]])
p,z,v=project_points([[1,2,0],[1,2,-3]],K0,np.eye(3),[0,0,0]);assert not v.any() and np.isnan(p).all()
assert project_points([],K0,np.eye(3),[0,0,0])[0].shape==(0,2)
# 2. Eight-point estimates scale-equivalent F and enforces rank2.
s=scene(99,noise=0,outlier_fraction=0,n=20);F=eight_point(s['p'],s['q'])
assert abs(np.linalg.det(F))<1e-9 and np.linalg.matrix_rank(F,tol=1e-8)==2
assert np.max(sampson_error(F,s['p'],s['q']))<1e-7
Fg=s['F']/np.linalg.norm(s['F']);assert abs(np.sum(F*Fg))>1-1e-6
F8=eight_point(s['p'][:8],s['q'][:8]);assert max(sampson_error(F8,s['p'],s['q']))<1e-6
shift=np.array([10000.,-20000.]);Fs=eight_point(s['p']+shift,s['q']+shift);assert max(sampson_error(Fs,s['p']+shift,s['q']+shift))<1e-5
for p,q in [(s['p'][:7],s['q'][:7]),(np.c_[np.arange(12),np.arange(12)],np.c_[np.arange(12),2*np.arange(12)]),(np.zeros((8,2)),np.zeros((8,2)))]:
    try:eight_point(p,q);raise AssertionError('Expected degeneracy error')
    except ValueError:pass
# 3. Horizontal epipolar geometry: sqrt Sampson = |dy|/sqrt(2), not |dy|.
Fh=np.array([[0.,0,0],[0,0,-1],[0,1,0]])
assert np.allclose(sampson_error(Fh,[[4,7]],[[10,9]]),[np.sqrt(2)])
assert np.allclose(sampson_error(Fh*7,[[4,7]],[[10,9]]),[np.sqrt(2)])
assert np.allclose(sampson_error(Fh*1e-12,[[4,7]],[[10,9]]),[np.sqrt(2)])
assert np.isinf(sampson_error(np.zeros((3,3)),[[0,0]],[[0,0]])).all()
assert sampson_error(Fh,[],[]).shape==(0,)
# 4. Clean data should need one trial; a robust noisy scene retains most GT inliers.
F,m,info=ransac_f(s['p'],s['q'],threshold=.01,seed=2);assert m.all() and info['trials']==1
rough=scene(198,noise=.5,outlier_fraction=.2,n=100);F,m,info=ransac_f(rough['p'],rough['q'],threshold=1.5,seed=4)
assert F is not None and (m&rough['inlier']).sum()>=60 and info['trials']<=1500
F,m,info=ransac_f(np.zeros((8,2)),np.zeros((8,2)),max_trials=5);assert F is None and not m.any() and info['trials']==5
# 5. DLT recovers 3D, reports infinite ray intersection, and leaves chirality to the caller.
P1=K0@np.c_[np.eye(3),np.zeros(3)];P2=K0@np.c_[np.eye(3),np.array([-.5,0,0])]
X,v=triangulate_dlt(P1,P2,[[480,320]],[[400,320]]);assert v[0] and np.allclose(X,[[1,.5,5]])
X,v=triangulate_dlt(P1,P2,[[400,300]],[[400,300]]);assert not v[0] and np.isnan(X).all()
X,v=triangulate_dlt(P1,P1,[[400,300]],[[400,300]]);assert not v[0]
X,v=triangulate_dlt(P1,P2,[[320,240]],[[400,240]]);assert v[0] and X[0,2]<0
assert not project_points(X,K0,np.eye(3),[0,0,0])[2][0]
assert triangulate_dlt(P1,P2,[],[])[0].shape==(0,3)
# Ablation preserves the sampled 3D points/noise source when only baseline changes.
a=scene(4100);b=scene(4100,baseline=.05);assert np.array_equal(a['X'],b['X']) and np.array_equal(a['p'],b['p']) and np.array_equal(a['inlier'],b['inlier'])
print('Open tests passed: camera, normalized F/rank2, Sampson units, RANSAC and triangulation degeneracies.')

## Собственные контрпримеры
Для проекции используйте другую камеру/точку, для F — масштабированные координаты, для ошибки — пару вне линии, для RANSAC — вырожденную выборку или большой процент выбросов, для триангуляции — точку за камерой либо почти параллельные лучи. Объясните, какие ошибки возвращают недоступный результат и почему это лучше правдоподобного числа.

In [ ]:
# TODO: ваши5входов, ожидаемые ответы и assert.

## Общий оценщик позы и глубины
Этот код дан готовым, чтобы все решения сравнивались одинаково. F сначала оценивается по наблюдениям. Внешний порог sqrt Sampson≤2 px одинаков для всех методов; precision/recall/F1 считают совпадение с заранее созданной GT-разметкой выбросов. Ошибка Sampson оценивается на **всех** GT-инлайерах, включая отвергнутые моделью.

После `E=K.T F K` проецируем E на допустимый спектр(s,s,0); recoverPose выбирает R,t по положительным глубинам. t имеет норму 1: это направление переноса, не baseline в метрах. Ошибка вращения и направления t считается в градусах, с учетом знака выбранного cheirality-решения.

Для глубины оценщик умножает реконструкцию на **известный синтетический baseline**: это внешняя информация для проверки, недоступная обычному монокулярному восстановлению. Медианная относительная ошибка Z считается на GT-инлайерах с положительной глубиной в двух камерах и углом>0.2°. Рядом обязательно depth_coverage — доля GT-инлайеров, для которых такая проверка прошла. Число pose_returned_cases означает наличие вычислимой позы, не доказательство правильности. Если оценки нет, поле None и status; такие случаи нельзя скрыть средним по оставшимся.

Planar, pure rotation и очень малый baseline—отдельные заранее построенные контрольные сценарии. Их именование в оценщике использует знание сценария, а не заявляет готовый детектор вырождения для реальных фотографий.

Для плоскости именно данный F-конвейер не оценивает позу и глубину. При известной K и модели гомографии движение и плоскость можно восстанавливать до масштаба с неоднозначностями; этот отдельный путь здесь не реализован. При чистом вращении отсутствует трансляционный базис для определения глубины.

In [ ]:
def median_finite(a):
    a=np.asarray(a);a=a[np.isfinite(a)]
    return float(np.median(a)) if len(a) else None

def score_model(F,s):
    p,q=s['p'],s['q'];truth=s['inlier'];empty={'status':'no_model','precision':0.,'recall':0.,'f1':0.,'accepted':0,'true_inliers':int(truth.sum()),'median_sampson_px':None,'rotation_deg':None,'translation_direction_deg':None,'depth_relative_error':None,'depth_coverage':0.}
    if not truth.any():
        return {**empty,'status':'no_gt_inliers','precision':None,'recall':None,'f1':None,'depth_coverage':None},None,None
    if F is None or not np.isfinite(F).all() or np.linalg.norm(F)<1e-14:return empty,None,None
    error=sampson_error(F,p,q);accepted=error<=2.;tp=int((accepted&truth).sum());precision=tp/max(1,int(accepted.sum()));recall=tp/int(truth.sum())
    result={**empty,'status':'geometry_not_yet_recovered','precision':precision,'recall':recall,'f1':2*precision*recall/max(1e-15,precision+recall),'accepted':int(accepted.sum()),'median_sampson_px':median_finite(error[truth])}
    # These are declared synthetic controls, not an automatic detector for arbitrary photographs.
    parallax=ray_angles(s['clean1'],s['clean2'],s['K'],s['R'])
    if s['kind']=='planar':result['status']='known_planar_control_pose_not_scored';return result,None,accepted
    if np.linalg.norm(s['t'])<1e-12:result['status']='pure_rotation_depth_unobservable';return result,None,accepted
    if np.median(parallax)<.2:result['status']='known_low_parallax_control';return result,None,accepted
    if accepted.sum()<8:result['status']='insufficient_consensus';return result,None,accepted
    E=s['K'].T@F@s['K'];U,sv,Vt=np.linalg.svd(E);E=U@np.diag([(sv[0]+sv[1])/2]*2+[0])@Vt
    try:
        count,R,t,pose_mask,_=cv2.recoverPose(E,p,q,s['K'],distanceThresh=1e6,mask=accepted.astype(np.uint8))
    except cv2.error:result['status']='pose_recovery_failed';return result,None,accepted
    if count<8:result['status']='insufficient_positive_depth';return result,None,accepted
    t=t.ravel();Rerr=np.degrees(np.arccos(np.clip((np.trace(R@s['R'].T)-1)/2,-1,1)))
    ttrue=s['t']/np.linalg.norm(s['t']);terr=np.degrees(np.arccos(np.clip(np.dot(t,ttrue)/np.linalg.norm(t),-1,1)))
    # recoverPose has ||t||=1. The unknown real baseline is used ONLY by the evaluator.
    P2unit=s['K']@np.c_[R,t];Xunit,finite=triangulate_dlt(s['P1'],P2unit,p,q)
    _,depth1,front1=project_points(Xunit,s['K'],np.eye(3),np.zeros(3));_,depth2,front2=project_points(Xunit,s['K'],R,t)
    estimated_angles=ray_angles(p,q,s['K'],R)
    valid=finite&front1&front2&(estimated_angles>.2)&truth
    metric_scale=float(np.linalg.norm(s['t']))
    Xmetric=Xunit*metric_scale
    relative=np.abs(Xmetric[:,2]-s['X'][:,2])/s['X'][:,2]
    result.update(status='ok',rotation_deg=float(Rerr),translation_direction_deg=float(terr),depth_relative_error=median_finite(relative[valid]),depth_coverage=float(valid.sum()/truth.sum()))
    return result,dict(R=R,t_unit=t,Cmetric=(-R.T@t)*metric_scale,Xmetric=Xmetric,valid=valid),accepted

def estimate(s,method,threshold,seed=41):
    if method=='F8_all':
        try:F=eight_point(s['p'],s['q']);return F,{'trials':1}
        except (ValueError,np.linalg.LinAlgError):return None,{'trials':1}
    F,mask,info=ransac_f(s['p'],s['q'],threshold=threshold,seed=seed)
    return F,info

def summarize(rows):
    result={'cases':len(rows),'pose_returned_cases':sum(r['status']=='ok' for r in rows)}
    for key in ['precision','recall','f1','median_sampson_px','rotation_deg','translation_direction_deg','depth_relative_error','depth_coverage','estimator_ms','trials']:
        values=[r[key] for r in rows if r.get(key) is not None]
        result['mean_'+key]=float(np.mean(values)) if values else None
    return result

def compare_figure(s,models,path):
    fig=plt.figure(figsize=(12,7),layout='constrained')
    for i,(method,F) in enumerate(models):
        metric,recovered,accepted=score_model(F,s);ax=fig.add_subplot(2,2,i+1)
        err=sampson_error(F,s['p'],s['q']) if F is not None else np.full(len(s['p']),np.inf)
        for truth,col,label in [(True,'#2563eb','GT inliers'),(False,'#ef4444','GT outliers')]:
            mask=s['inlier']==truth;ax.scatter(np.flatnonzero(mask),np.minimum(err[mask],1e3),s=12,c=col,label=label)
        ax.axhline(2,c='k',ls='--');ax.set_yscale('symlog',linthresh=1);ax.set_ylim(bottom=0);ax.set(title=f'{method}: P={metric["precision"]:.2f}, R={metric["recall"]:.2f}',xlabel='same pair index',ylabel='Sqrt Sampson (px; clipped at 1000)');ax.legend()
        ax=fig.add_subplot(2,2,i+3,projection='3d');draw_scene(ax,s)
        if recovered is not None:
            X=recovered['Xmetric'][recovered['valid']];ax.scatter(X[:,0],X[:,2],X[:,1],c='#f59e0b',s=16,marker='x')
            C=recovered['Cmetric'];ax.scatter(C[0],C[2],C[1],c='#f59e0b',s=60,marker='^')
            ax.set_title(f'{method}: orange reconstruction\nGT baseline supplied for scoring',fontsize=10)
        else:ax.set_title(method+': '+metric['status'])
    fig.savefig(path,dpi=140);plt.show();plt.close(fig)

## Val: выбираем только один fitting threshold
F8_all—нормализованный восьмиточечный алгоритм по всем наблюдениям, без удаления выбросов. RANSAC8 использует ваш тот же решатель на случайных подвыборках. На val 5 независимых seeds и thresholds .5/1/2/4 px. Настройка выбирается по максимальному среднему F1 внешнего 2 px оценщика; tie—меньший порог. Порог оценки по результатам не меняется.

In [ ]:
RESULTS=Path('results');RESULTS.mkdir(exist_ok=True)
VAL_SEEDS=[3100,3101,3102,3103,3104]
TEST_SEEDS=[4100,4101,4102,4103,4104]
THRESHOLDS=[.5,1.,2.,4.]
val_rows=[]
for seed in VAL_SEEDS:
    s=scene(seed)
    for threshold in THRESHOLDS:
        start=time.perf_counter();F,info=estimate(s,'RANSAC8',threshold);elapsed=(time.perf_counter()-start)*1000
        metrics,_,_=score_model(F,s)
        val_rows.append({'seed':seed,'method':'RANSAC8','threshold':threshold,**metrics,'estimator_ms':elapsed,'trials':info['trials']})
save_csv(RESULTS/'validation.csv',val_rows)
val_summary=[{'threshold':t,**summarize([r for r in val_rows if r['threshold']==t])} for t in THRESHOLDS]
(RESULTS/'validation_summary.json').write_text(json.dumps(val_summary,indent=2,allow_nan=False))
for row in val_summary:print(row)
fig,axs=plt.subplots(1,2,figsize=(10,3.5),layout='constrained')
for key,label in [('mean_precision','precision'),('mean_recall','recall'),('mean_f1','F1')]:axs[0].plot(THRESHOLDS,[r[key] for r in val_summary],'-o',label=label)
axs[0].set(xlabel='Fitting sqrt-Sampson threshold (px)',ylabel='Common 2px evaluation',ylim=(0,1.05));axs[0].legend()
axs[1].plot(THRESHOLDS,[r['mean_trials'] for r in val_summary],'-o');axs[1].set(xlabel='Fitting threshold (px)',ylabel='Mean sampled hypotheses');fig.savefig(RESULTS/'validation.png',dpi=140);plt.show();plt.close(fig)

## Перед фиксацией
Запишите выбранный threshold и обоснование. Сопоставьте F1, число гипотез и ошибки на всех GT-инлайерах. Если хороший F1 не дал хорошей позы, объясните ограничения F-ошибки и парных наблюдений. Не открывайте test до этой записи.

In [ ]:
# Announced before test: maximize mean val F1 at common 2 px; tie -> smaller threshold.
chosen=max(val_summary,key=lambda r:(r['mean_f1'],-r['threshold']))
locked={'threshold':chosen['threshold'],'val_seeds':VAL_SEEDS,'test_seeds':TEST_SEEDS,'scene_points':160,'noise_px':.7,'outlier_fraction':.25,'baseline_m':.5,'max_trials':1500,'confidence_requested':.999,'external_threshold_px':2.,'rule':'max mean val F1; tie smaller threshold','test_opened':False}
(RESULTS/'selection_locked.json').write_text(json.dumps(locked,indent=2,allow_nan=False))
print('Frozen fit threshold:',locked['threshold'],'px. Evaluation stays2px.')

## Test и заранее заданная абляция
Test использует 5 новых seeds. Сравните F8_all и RANSAC8 на одинаковых наблюдениях. Время измеряется только для оценки F: один прогрев и три повтора; simulation, score, recoverPose, triangulation и рисунки исключены. Отдельно оцените стоимость полного notebook на своем CPU.

Абляция меняет только baseline .5→.05 м при тех же 3D точках, seeds, шуме, выбросах и frozen threshold . Входные пиксели второго кадра меняются физически; это опыт по наблюдаемости глубины. Не выбирайте новый threshold по short-baseline test.

Две одинаковые пары показываются для обоих методов: ошибки всех наблюдений и реконструкция с GT-масштабом. Проанализируйте успехи и ошибки, их нельзя заменить одним средним F1.

In [ ]:
RUN_TEST=False

In [ ]:
if RUN_TEST:
    locked=json.loads((RESULTS/'selection_locked.json').read_text());rows=[];timing=[]
    for seed in TEST_SEEDS:
        s=scene(seed);models=[]
        for method in ['F8_all','RANSAC8']:
            times=[]
            for repeat in range(4):
                start=time.perf_counter();F,info=estimate(s,method,locked['threshold']);ms=(time.perf_counter()-start)*1000
                if repeat:times.append(ms);timing.append({'seed':seed,'method':method,'repeat':repeat,'estimator_ms':ms})
            metrics,_,_=score_model(F,s);rows.append({'seed':seed,'method':method,**metrics,'estimator_ms':float(np.median(times)),'trials':info['trials']});models.append((method,F))
        if seed in [TEST_SEEDS[0],TEST_SEEDS[-1]]:compare_figure(s,models,RESULTS/f'comparison_{seed}.png')
        # Predeclared ablation: only change physical baseline .5 -> .05m;
        # same X distribution/random seed, pixel noise and outliers, same frozen threshold.
        short=scene(seed,baseline=.05);F,info=estimate(short,'RANSAC8',locked['threshold']);metric,_,_=score_model(F,short)
        rows.append({'seed':seed,'method':'RANSAC8_baseline_0.05m',**metric,'estimator_ms':None,'trials':info['trials']})
    save_csv(RESULTS/'test.csv',rows);save_csv(RESULTS/'timing_repeats.csv',timing)
    summary=[{'method':m,**summarize([r for r in rows if r['method']==m])} for m in ['F8_all','RANSAC8','RANSAC8_baseline_0.05m']]
    (RESULTS/'test_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False))
    locked['test_opened']=True;(RESULTS/'selection_locked.json').write_text(json.dumps(locked,indent=2,allow_nan=False))
    env={'python':platform.python_version(),'numpy':np.__version__,'opencv':cv2.__version__,'opencv_threads':cv2.getNumThreads(),'cpu':next((line.split(':',1)[1].strip() for line in Path('/proc/cpuinfo').read_text().splitlines() if line.startswith('model name')),platform.machine()) if Path('/proc/cpuinfo').exists() else platform.machine(),'points':160,'warmup':1,'timing_repeats':3,'timing_scope':'F estimation only; excludes simulation, score, recoverPose, triangulation, figures','translation_scale':'recoverPose t norm1; GT baseline supplied only to depth evaluation'}
    (RESULTS/'environment.json').write_text(json.dumps(env,indent=2))
    for item in summary:print(item)
else:print('Test closed: interpret val, freeze choice, then set RUN_TEST=True.')

## Самостоятельная диагностика вырождения
После основного test создайте три дополнительных сцены: `kind='planar'`, `kind='pure_rotation'`, baseline=.002 м. Это диагностические примеры вне выбора модели. Используйте тот же threshold . Покажите камеры/лучи и объясните, почему малый residual или высокая доля accepted не доказывают корректные глубины. Добавьте **свой** контрольный случай, например коллинеарные наблюдения; не скрывайте None/неудачу.

## Сдача
- Таблица по девяти стендам: три пункта на стенд с прогнозом, измерением и объяснением.
- Пять функций, открытые проверки и 5 собственных ручных примеров.
- Val, зафиксированная настройка, test двух методов и абляция, CSV/JSON.
- Одинаковые случаи с ошибками/камерами/3D, три вырождения и свой случай.
- Время, версии, доля успешных вычислений и ограничения масштаба/глубины.

Свяжите результат со следующей темой: для SfM нужны общие камеры и согласованные tracks, а не набор независимых пар с несогласованными масштабами. Перед распространением notebook очистите свои outputs.

## Источники

- [Hartley & Zisserman, Multiple View Geometry, глава 9](https://www.robots.ox.ac.uk/~vgg/hzbook/hzbook2/HZepipolar.pdf): `x2.T @ F @ x1=0`, F/E и эпиполярные линии.
- [Hartley, In Defense of the Eight-Point Algorithm](https://users.cecs.anu.edu.au/~hartley/Papers/fundamental/fundamental.pdf): нормализация координат для линейной оценки F.
- [OpenCV4.12: Camera Calibration and 3D Reconstruction](https://docs.opencv.org/4.12.0/d9/d0c/group__calib3d.html): projection, findFundamentalMat, recoverPose, triangulatePoints. `recoverPose` возвращает направление t с нормой 1; baseline в метрах из двух монокулярных кадров не определяется.
- [Fischler & Bolles, RANSAC, 1981](https://doi.org/10.1145/358669.358692): случайные гипотезы и согласованные наблюдения.

Все рисунки вычислены учебным кодом на авторской синтетике. Они не являются фотографиями, измеренным benchmark или воспроизведением полной SfM-системы.